# Week 8: Dockerized End-to-End API Verification

This notebook smoke-tests the running **arXiv Paper Curator FastAPI service** after the Docker Compose stack has been built and started. It covers the original retrieval endpoints and the newer metadata discovery, Project Match, Research Radar, paper-specific RAG, preparation, and Deep Dive endpoints.

**Important:** this notebook sends real HTTP requests to `BASE_URL`. It does not build or start Docker itself, and it does not deploy anything. Run it only after you have intentionally started the stack. Start with the inexpensive checks; enable the optional expensive flows only when you are ready.

## What it checks

- Service health and runtime route registration through OpenAPI.
- Metadata discovery, Project-to-Paper Match, and Research Radar.
- BM25-only hybrid-search endpoint behavior without invoking embeddings.
- API input validation.
- Optional traditional RAG, Agentic RAG, and streaming generation.
- Optional one-paper prepare → paper-scoped ask → structured Deep Dive flow.
- Optional feedback submission and full paper JSON endpoints.

A successful HTTP response proves the endpoint responded for this deployment; it does **not** prove research relevance or correctness of generated answers. Review the returned papers and evidence manually.


## 1. Start Docker Compose before running the notebook

Run these commands in a terminal from the repository root (not in a notebook cell):

```bash
docker compose up --build -d
docker compose ps
docker compose logs --tail=100 api
```

Check that the API, PostgreSQL, OpenSearch, and Redis containers are healthy. For generation tests, check that the model is available inside the Ollama container:

```bash
docker compose exec ollama ollama list
# If the configured model is not present, pull it explicitly:
docker compose exec ollama ollama pull qwen2.5-coder:1.5b
```

The notebook's default target is `http://localhost:8000`, for a notebook running on the same machine as Docker. If testing a remote server, prefer an SSH tunnel (for example, `ssh -L 8000:127.0.0.1:8000 USER@SERVER`) and keep `BASE_URL` set to localhost through that tunnel. Do **not** expose the unauthenticated API or the OpenSearch port publicly just for this test.

The application currently has no API authentication. Do not put passwords, API keys, or other secrets into this notebook.


In [ ]:
import os
import json
import time
from urllib.parse import quote

import requests

# Set ARXIV_API_BASE_URL in the environment, or change this default.
BASE_URL = os.getenv("ARXIV_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_PREFIX = "/api/v1"
MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:1.5b")
DEFAULT_TIMEOUT = 30
GENERATION_TIMEOUT = 240

# Conservative defaults: inexpensive smoke tests run first.
# Flip a flag to True only when you intentionally want that operation to run.
RUN_GENERATION_TESTS = False          # Calls local Ollama; may take several minutes.
RUN_STREAMING_TEST = False            # Calls the streaming RAG generation endpoint.
RUN_HYBRID_EMBEDDINGS_TEST = False    # May call the configured embedding provider (Jina).
RUN_PAPER_PREPARATION_FLOW = False    # Downloads/parses one PDF if needed, embeds chunks, and writes DB/OpenSearch data.
RUN_PAPER_DETAIL_TEST = False         # Full paper responses may contain large raw_text/sections fields.
RUN_PAPER_LIST_TEST = False           # The list endpoint may return large parsed-content fields.
RUN_FEEDBACK_TEST = False             # Writes feedback to Langfuse if a trace_id is available.

checks = []
print("BASE_URL:", BASE_URL)
print("Model requested by optional generation tests:", MODEL)
print("Current flags:", {
    "generation": RUN_GENERATION_TESTS,
    "streaming": RUN_STREAMING_TEST,
    "hybrid_embeddings": RUN_HYBRID_EMBEDDINGS_TEST,
    "paper_preparation_flow": RUN_PAPER_PREPARATION_FLOW,
    "paper_detail": RUN_PAPER_DETAIL_TEST,
    "paper_list": RUN_PAPER_LIST_TEST,
    "feedback_write": RUN_FEEDBACK_TEST,
})


In [ ]:
def record_check(name, status, detail=""):
    # Record a PASS, WARN, FAIL, or SKIP without stopping the whole notebook.
    row = {"check": name, "status": status, "detail": str(detail)[:500]}
    checks.append(row)
    print(f"[{status}] {name}" + (f" — {row['detail']}" if row["detail"] else ""))


def api_call(name, method, path, *, payload=None, params=None, expected=(200,), timeout=None):
    # Call an endpoint, record its HTTP result, and return (response, parsed body).
    url = BASE_URL + path
    try:
        response = requests.request(
            method=method,
            url=url,
            json=payload,
            params=params,
            timeout=timeout or DEFAULT_TIMEOUT,
            headers={"Accept": "application/json"},
        )
    except requests.RequestException as exc:
        record_check(name, "FAIL", f"Request could not reach endpoint: {type(exc).__name__}: {exc}")
        return None, None

    try:
        body = response.json()
    except ValueError:
        body = {"raw_text": response.text[:1000]}

    status = "PASS" if response.status_code in expected else "FAIL"
    record_check(name, status, f"HTTP {response.status_code}; expected {list(expected)}")
    if status == "FAIL":
        print("  Response sample:", json.dumps(body, ensure_ascii=False, default=str)[:1000])
    return response, body


def short_paper_summary(paper):
    # Print compact fields without dumping full abstracts or parsed text.
    return {
        "arxiv_id": paper.get("arxiv_id"),
        "title": paper.get("title"),
        "published_date": paper.get("published_date"),
        "categories": paper.get("categories"),
        "score": paper.get("search_score", paper.get("score", paper.get("fusion_score"))),
    }


## 2. Service health and runtime route registration

The health endpoint checks API dependencies including PostgreSQL, OpenSearch, and Ollama. An overall `degraded` response is useful diagnostic information even if the HTTP status is 200. The OpenAPI check confirms the routes are registered in the running build, rather than only existing in source files.


In [ ]:
health_response, health = api_call("API health endpoint", "GET", API_PREFIX + "/health")
if isinstance(health, dict):
    print("Overall status:", health.get("status"))
    print("Service statuses:")
    print(json.dumps(health.get("services", {}), indent=2, ensure_ascii=False, default=str))
    if health.get("status") != "ok":
        record_check("Dependency readiness", "WARN", "Health response is not fully ok; inspect service statuses above.")
    else:
        record_check("Dependency readiness", "PASS", "Health response reports all checked dependencies healthy.")

openapi_response, openapi = api_call("OpenAPI schema", "GET", "/openapi.json")
expected_routes = {
    ("get", "/api/v1/health"),
    ("post", "/api/v1/ask"),
    ("post", "/api/v1/stream"),
    ("post", "/api/v1/search/"),
    ("post", "/api/v1/ask-agentic"),
    ("post", "/api/v1/feedback"),
    ("get", "/api/v1/papers/discover"),
    ("post", "/api/v1/papers/match-project"),
    ("post", "/api/v1/papers/radar"),
    ("post", "/api/v1/papers/{arxiv_id}/prepare"),
    ("post", "/api/v1/papers/{arxiv_id}/deep-dive"),
    ("post", "/api/v1/papers/{arxiv_id}/ask"),
    ("get", "/api/v1/papers/{arxiv_id}"),
}
if isinstance(openapi, dict):
    path_map = openapi.get("paths", {})
    missing_routes = sorted(f"{method.upper()} {path}" for method, path in expected_routes
                            if method not in path_map.get(path, {}))
    if missing_routes:
        record_check("Expected runtime routes registered", "FAIL", "Missing: " + ", ".join(missing_routes))
    else:
        record_check("Expected runtime routes registered", "PASS", f"All {len(expected_routes)} checked routes appear in OpenAPI.")
else:
    record_check("Expected runtime routes registered", "SKIP", "OpenAPI schema was not available.")


## 3. Metadata-only endpoints

These requests do not intentionally download or parse PDFs, call an LLM, or create embeddings. Empty search results can be valid; a 5xx response or an error-shaped body is not.


In [ ]:
# Metadata discovery
_discovery_response, discovery = api_call(
    "Metadata discovery (BM25 metadata index)",
    "GET",
    API_PREFIX + "/papers/discover",
    params={"query": "transformer attention", "size": 5, "offset": 0},
)
metadata_papers = discovery.get("papers", []) if isinstance(discovery, dict) else []
if isinstance(discovery, dict) and isinstance(metadata_papers, list):
    record_check("Metadata discovery response shape", "PASS", f"total={discovery.get('total')}; returned={len(metadata_papers)}")
    for paper in metadata_papers[:5]:
        print(short_paper_summary(paper))
else:
    record_check("Metadata discovery response shape", "FAIL", "Expected a JSON object containing a papers list.")

# If the query found no candidate, fetch the newest indexed metadata records for later optional paper-scoped tests.
if not metadata_papers and _discovery_response is not None and _discovery_response.status_code == 200:
    _, newest = api_call(
        "Metadata discovery fallback (latest records)", "GET", API_PREFIX + "/papers/discover",
        params={"query": "", "size": 5, "offset": 0},
    )
    if isinstance(newest, dict):
        metadata_papers = newest.get("papers", []) or []

selected_paper = metadata_papers[0] if metadata_papers else None
selected_arxiv_id = selected_paper.get("arxiv_id") if selected_paper else None
if selected_paper:
    print("Selected exact arXiv ID for optional paper-scoped checks:", selected_arxiv_id)
    print("Selected title:", selected_paper.get("title", ""))
else:
    record_check("Select paper for scoped tests", "SKIP", "Metadata index returned no papers. Populate/sync metadata before paper-scoped testing.")


In [ ]:
# Project-to-Paper Match
project_payload = {
    "project_title": "Evidence-grounded research assistant",
    "project_description": "Build a retrieval augmented generation assistant that searches scientific papers and answers technical questions using evidence from the papers.",
    "technologies": ["RAG", "vector search", "large language models"],
    "research_interests": ["retrieval augmented generation", "scientific information retrieval", "LLM evaluation"],
    "categories": ["cs.AI", "cs.CL", "cs.IR"],
    "top_k": 5,
}
_project_response, project_result = api_call(
    "Project-to-Paper Match", "POST", API_PREFIX + "/papers/match-project", payload=project_payload
)
if isinstance(project_result, dict) and isinstance(project_result.get("papers"), list):
    record_check("Project Match response shape", "PASS", f"result_count={project_result.get('result_count')}; returned={len(project_result['papers'])}")
    for paper in project_result["papers"][:5]:
        print(short_paper_summary(paper))
else:
    record_check("Project Match response shape", "FAIL", "Expected a JSON object containing a papers list.")

# Research Radar
radar_payload = {
    "topics": ["retrieval augmented generation", "LLM evaluation", "scientific information retrieval"],
    "categories": ["cs.AI", "cs.CL", "cs.IR"],
    "lookback_days": 365,
    "top_k": 5,
    "per_topic_candidate_limit": 10,
}
_radar_response, radar_result = api_call(
    "Research Radar", "POST", API_PREFIX + "/papers/radar", payload=radar_payload
)
if isinstance(radar_result, dict) and isinstance(radar_result.get("papers"), list):
    record_check("Research Radar response shape", "PASS", f"result_count={radar_result.get('result_count')}; warnings={len(radar_result.get('warnings', []))}")
    for paper in radar_result["papers"][:5]:
        print(short_paper_summary(paper), "matched_topics=", paper.get("matched_topics"))
    if radar_result.get("warnings"):
        print("Topic-level warnings:", json.dumps(radar_result["warnings"], indent=2, ensure_ascii=False))
else:
    record_check("Research Radar response shape", "FAIL", "Expected a JSON object containing a papers list.")

# BM25-only search keeps this inexpensive and avoids embedding-provider calls.
_search_response, search_result = api_call(
    "Hybrid-search endpoint in BM25 mode", "POST", API_PREFIX + "/search/",
    payload={"query": "transformer attention", "size": 5, "from": 0,
             "categories": ["cs.AI", "cs.CL"], "latest_papers": False,
             "use_hybrid": False, "min_score": 0.0},
)
if isinstance(search_result, dict) and isinstance(search_result.get("hits"), list):
    record_check("Search response shape", "PASS", f"total={search_result.get('total')}; returned={len(search_result['hits'])}; mode={search_result.get('search_mode')}")
    for hit in search_result["hits"][:5]:
        print(short_paper_summary(hit))
else:
    record_check("Search response shape", "FAIL", "Expected a JSON object containing a hits list.")


## 4. Validation checks (no generation)

These requests intentionally submit invalid payloads and should return HTTP 422. They are safe negative tests and do not require an indexed corpus.


In [ ]:
api_call(
    "Project Match rejects empty project description", "POST", API_PREFIX + "/papers/match-project",
    payload={"project_description": ""}, expected=(422,)
)
api_call(
    "Research Radar rejects empty topic list", "POST", API_PREFIX + "/papers/radar",
    payload={"topics": []}, expected=(422,)
)
api_call(
    "Traditional RAG rejects empty query", "POST", API_PREFIX + "/ask",
    payload={"query": ""}, expected=(422,)
)
api_call(
    "Project Match rejects inverted date range", "POST", API_PREFIX + "/papers/match-project",
    payload={"project_description": "Test project", "date_from": "2026-10-22", "date_to": "2020-01-01"},
    expected=(422,)
)


## 5. Optional hybrid embedding test

Enable `RUN_HYBRID_EMBEDDINGS_TEST = True` in the setup cell to test the vector/hybrid path. This can call the configured embedding provider and may require valid credentials and network access from the container.


In [ ]:
if RUN_HYBRID_EMBEDDINGS_TEST:
    _hybrid_response, hybrid_result = api_call(
        "Hybrid search with embeddings enabled", "POST", API_PREFIX + "/search/",
        payload={"query": "retrieval augmented generation evaluation", "size": 5, "from": 0,
                 "categories": ["cs.AI", "cs.CL"], "latest_papers": False,
                 "use_hybrid": True, "min_score": 0.0},
        timeout=GENERATION_TIMEOUT,
    )
    if isinstance(hybrid_result, dict):
        print("Reported search mode:", hybrid_result.get("search_mode"))
        print("Returned hits:", len(hybrid_result.get("hits", [])))
else:
    record_check("Hybrid embedding search", "SKIP", "Set RUN_HYBRID_EMBEDDINGS_TEST=True to run the embedding-backed path.")


## 6. Optional traditional RAG, Agentic RAG, and streaming RAG

These tests invoke model generation. Keep them disabled during the first dependency/route check if Ollama is not ready. Enable them once health checks look reasonable and the requested model is installed. `use_hybrid=False` avoids the query-embedding call, but generation still consumes local compute.


In [ ]:
agentic_result = None
if RUN_GENERATION_TESTS:
    _ask_response, ask_result = api_call(
        "Traditional RAG generation", "POST", API_PREFIX + "/ask",
        payload={"query": "Using retrieved papers, explain self-attention in transformer models.",
                 "top_k": 3, "use_hybrid": False, "model": MODEL,
                 "categories": ["cs.AI", "cs.CL"]},
        timeout=GENERATION_TIMEOUT,
    )
    if isinstance(ask_result, dict):
        print("Answer preview:", str(ask_result.get("answer", ""))[:1200])
        print("Sources:", ask_result.get("sources", []))

    _agentic_response, agentic_result = api_call(
        "Agentic RAG generation", "POST", API_PREFIX + "/ask-agentic",
        payload={"query": "How do retrieval augmented generation systems reduce unsupported answers?",
                 "top_k": 3, "use_hybrid": False, "model": MODEL,
                 "categories": ["cs.AI", "cs.CL"]},
        timeout=GENERATION_TIMEOUT,
    )
    if isinstance(agentic_result, dict):
        print("Agentic answer preview:", str(agentic_result.get("answer", ""))[:1200])
        print("Reasoning steps:", agentic_result.get("reasoning_steps", []))
        print("Retrieval attempts:", agentic_result.get("retrieval_attempts"))
        print("Trace ID:", agentic_result.get("trace_id"))

    if RUN_FEEDBACK_TEST and isinstance(agentic_result, dict) and agentic_result.get("trace_id"):
        api_call(
            "Optional Langfuse feedback write", "POST", API_PREFIX + "/feedback",
            payload={"trace_id": agentic_result["trace_id"], "score": 1.0,
                     "comments": "Smoke test from the Docker API verification notebook."},
            timeout=DEFAULT_TIMEOUT,
        )
    elif RUN_FEEDBACK_TEST:
        record_check("Optional Langfuse feedback write", "SKIP", "No trace_id returned by Agentic RAG.")
else:
    record_check("Traditional RAG generation", "SKIP", "Set RUN_GENERATION_TESTS=True when Ollama/model are ready.")
    record_check("Agentic RAG generation", "SKIP", "Set RUN_GENERATION_TESTS=True when Ollama/model are ready.")
    if RUN_FEEDBACK_TEST:
        record_check("Optional Langfuse feedback write", "SKIP", "Generation tests are disabled.")

if RUN_STREAMING_TEST:
    try:
        stream_response = requests.post(
            BASE_URL + API_PREFIX + "/stream",
            json={"query": "Summarize the role of attention in transformer models.",
                  "top_k": 3, "use_hybrid": False, "model": MODEL},
            headers={"Accept": "text/event-stream", "Content-Type": "application/json"},
            timeout=GENERATION_TIMEOUT,
        )
        record_check("Streaming RAG HTTP response", "PASS" if stream_response.status_code == 200 else "FAIL",
                     f"HTTP {stream_response.status_code}; content-type={stream_response.headers.get('content-type')}")
        print("Stream response preview:\n", stream_response.text[:2000])
    except requests.RequestException as exc:
        record_check("Streaming RAG HTTP response", "FAIL", f"{type(exc).__name__}: {exc}")
else:
    record_check("Streaming RAG", "SKIP", "Set RUN_STREAMING_TEST=True to call the streaming generation endpoint.")


## 7. Optional selected-paper flow: prepare → scoped ask → Deep Dive

This is the stateful/expensive test. Set `RUN_PAPER_PREPARATION_FLOW = True` only when you are ready to test a single selected paper end to end.

It may download a PDF, run Docling, store parsed content in PostgreSQL, create chunk embeddings through the configured embedding service, and write the paper's chunks to OpenSearch. It will only request preparation for one exact arXiv ID selected from metadata discovery; it does not launch bulk ingestion. If preparation succeeds, the notebook tests the paper-scoped ask and structured Deep Dive with the **same exact ID**.

If the metadata index is empty or out of sync with PostgreSQL, choose an ID that is already stored in PostgreSQL and present in metadata discovery. Never use an ID copied from a different service if the exact version is not stored.


In [ ]:
if RUN_PAPER_PREPARATION_FLOW and selected_arxiv_id:
    encoded_id = quote(selected_arxiv_id, safe="")
    paper_path = f"{API_PREFIX}/papers/{encoded_id}"

    _prepare_response, prepare_result = api_call(
        "Prepare selected paper (downloads/parses/indexes if needed)",
        "POST", paper_path + "/prepare", payload=None, timeout=GENERATION_TIMEOUT,
    )
    if _prepare_response is not None and _prepare_response.status_code == 200:
        print("Preparation summary:", json.dumps(prepare_result, indent=2, ensure_ascii=False, default=str)[:1500])

        _paper_ask_response, paper_ask_result = api_call(
            "Paper-scoped RAG ask", "POST", paper_path + "/ask",
            payload={"query": "What problem does this paper address, and what evidence supports its main contribution?",
                     "top_k": 4, "use_hybrid": False, "model": MODEL},
            timeout=GENERATION_TIMEOUT,
        )
        if isinstance(paper_ask_result, dict):
            print("Scoped answer preview:", str(paper_ask_result.get("answer", ""))[:1200])
            print("Exact paper ID returned:", paper_ask_result.get("arxiv_id"))
            print("Evidence count:", len(paper_ask_result.get("evidence", [])))

        _deep_dive_response, deep_dive_result = api_call(
            "Structured paper Deep Dive", "POST", paper_path + "/deep-dive",
            payload={"model": MODEL, "top_k": 5, "use_hybrid": False},
            timeout=GENERATION_TIMEOUT,
        )
        if isinstance(deep_dive_result, dict):
            print("Deep Dive cache hit:", deep_dive_result.get("cached"))
            brief = deep_dive_result.get("brief", {})
            print("Executive summary:", str(brief.get("executive_summary", ""))[:1000])
            print("Evidence references:", len(brief.get("evidence_references", [])))
            print("Information gaps:", brief.get("information_gaps", []))
    else:
        record_check("Paper-scoped ask after preparation", "SKIP", "Preparation did not return HTTP 200; inspect its explicit error above.")
        record_check("Deep Dive after preparation", "SKIP", "Preparation did not return HTTP 200; inspect its explicit error above.")
elif RUN_PAPER_PREPARATION_FLOW and not selected_arxiv_id:
    record_check("Paper preparation flow", "SKIP", "No candidate ID available from metadata discovery.")
else:
    record_check("Paper preparation + scoped ask + Deep Dive", "SKIP", "Set RUN_PAPER_PREPARATION_FLOW=True to run the selected-paper, stateful flow.")


## 8. Optional large paper JSON endpoints

The existing `GET /api/v1/papers/` and `GET /api/v1/papers/{arxiv_id}` responses include optional parsed fields such as `raw_text`, `sections`, and `references`. On heavily processed papers these payloads can be large, so these calls are disabled by default. The code below prints only a compact summary, not the full body.


In [ ]:
if RUN_PAPER_LIST_TEST:
    _list_response, list_result = api_call(
        "Paper list endpoint (may return large parsed content)", "GET", API_PREFIX + "/papers/",
        params={"limit": 1, "offset": 0}, timeout=GENERATION_TIMEOUT,
    )
    if isinstance(list_result, dict):
        print("Paper list total:", list_result.get("total"))
        first = (list_result.get("papers") or [None])[0]
        if first:
            print({k: first.get(k) for k in ("arxiv_id", "title", "pdf_processed", "parser_used")})
else:
    record_check("Paper list endpoint", "SKIP", "Set RUN_PAPER_LIST_TEST=True; the payload can contain full parsed text.")

if RUN_PAPER_DETAIL_TEST and selected_arxiv_id:
    encoded_id = quote(selected_arxiv_id, safe="")
    _detail_response, detail_result = api_call(
        "Paper details endpoint (may return large parsed content)", "GET", f"{API_PREFIX}/papers/{encoded_id}",
        timeout=GENERATION_TIMEOUT,
    )
    if isinstance(detail_result, dict):
        print({k: detail_result.get(k) for k in ("arxiv_id", "title", "pdf_processed", "parser_used", "pdf_processing_date")})
else:
    record_check("Paper details endpoint", "SKIP", "Set RUN_PAPER_DETAIL_TEST=True after a candidate has been selected.")


## 9. Results summary

Review failures and warnings rather than relying only on the total. A 503 from Project Match or Radar can indicate metadata-index setup/sync issues; generation errors can indicate an unavailable Ollama model; preparation errors can indicate download, parsing, embedding, database, or OpenSearch problems. Check `docker compose logs --tail=200 api` and the relevant dependency container logs before rerunning expensive calls.


In [ ]:
from collections import Counter

counts = Counter(row["status"] for row in checks)
print("CHECK SUMMARY")
print("=" * 72)
for row in checks:
    print(f"{row['status']:4} | {row['check']}" + (f" | {row['detail']}" if row["detail"] else ""))
print("-" * 72)
print("Totals:", dict(counts))
print("A smoke test is not a relevance benchmark. Review retrieved papers, answers, and evidence manually.")


## Troubleshooting notes

- **Health is degraded:** inspect the per-service health JSON. Ollama may be reachable while the requested model is missing.
- **Project Match / Radar returns 503:** verify the metadata index exists and has documents; a working chunk index does not guarantee the metadata index has been synchronized.
- **Search is empty but returns 200:** the index may be empty, filters may be strict, or the query may not match the corpus. Compare with `GET /api/v1/papers/discover?query=&size=5`.
- **Paper prepare returns 404:** the exact selected version may be in the metadata index but absent from PostgreSQL; verify the exact versioned ID and indexing synchronization.
- **Paper ask returns 409:** the paper exists but its full text chunks are not indexed. Run the opt-in preparation flow for that exact ID.
- **Deep Dive returns 422 `insufficient_evidence`:** preparation may have succeeded but retrieval did not find enough evidence for the requested paper.
- **Hybrid mode silently reports BM25/fails:** verify embedding provider configuration and service-to-service network access.
- **Runtime route missing from OpenAPI:** rebuild the API image and verify the running container uses the intended source revision.

This notebook intentionally does not run Docker commands, trigger daily Airflow ingestion, delete data, or bulk-process the corpus.
